# Module 06 — LLM Evaluation

**Offline educational demonstration.** Evaluate retrieval with labeled question-answer pairs.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
p = Path('data')
if not p.exists(): p = Path('../data')
faq = pd.read_csv(p/'helpdesk_faq.csv')
labels = pd.read_csv(p/'evaluation_queries.csv')
print(faq[['doc_id','category','question']].head())


## Build and inspect a local TF-IDF retrieval index

In [ ]:
texts = (faq['question']+' '+faq['answer']).tolist()
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
matrix = vectorizer.fit_transform(texts)
def search(query, k=3, category=None):
    candidates = faq.index if category is None else faq.index[faq['category'].eq(category)]
    if len(candidates)==0: return pd.DataFrame(columns=['doc_id','question','answer','score'])
    scores = cosine_similarity(vectorizer.transform([query]), matrix[candidates]).ravel()
    ix = np.argsort(-scores)[:k]
    result = faq.loc[candidates[ix], ['doc_id','question','answer']].copy()
    result['score'] = scores[ix]
    return result.reset_index(drop=True)
print(search('How can I track my delivery?'))


In [ ]:
def hit_at_k(k=3):
    return np.mean([row.relevant_doc_id in set(search(row.query,k=k)['doc_id']) for row in labels.itertuples(index=False)])
print('Hit@1:',hit_at_k(1),'Hit@3:',hit_at_k(3))
for row in labels.itertuples(index=False):
    pred=search(row.query,1).iloc[0]['doc_id']
    if pred!=row.relevant_doc_id:print('Miss:',row.query,'expected',row.relevant_doc_id,'got',pred)

## Reflection
Explain one failure mode, one measurement you would track, and how the demo differs from a production LLM system.